## Kernel Implementations


In [5]:
import os
os.environ["JAX_ENABLE_X64"] = "True"
os.environ["JAX_PLATFORMS"] = "gpu"

import numpy as np
import matplotlib.pyplot as plt
import sys
import time
sys.path.append("../..")

from src_jax.analytic_kernel import AnalyticKernel
from src_jax.numerical_kernel import NumericalKernel

### Compare numerical and analytic approaches to computing the kernel

In [6]:
# NumericalKernel always requires nspot (total count) for forward simulations.
TSIM = 300.0
nspot = 40
nspot_rate = nspot / TSIM   # spots/day

theta_numerical = dict(peq=3.0,
                        kappa=0.3,
                        inc=np.pi/2,
                        nspot=nspot,
                        lspot=10.0,
                        tau=5.0,
                        alpha_max=0.05,
                        fspot=0.)

# AnalyticKernel uses nspot_rate (preferred) — sigma_k computed automatically.
# nspot_rate = sqrt-rate of spot emergence [spots/day]
theta_analytic = dict(peq=3.0,
                       kappa=0.3,
                       inc=np.pi/2,
                       nspot_rate=nspot_rate,
                       lspot=10.0,
                       tau=5.0,
                       alpha_max=0.05,
                       fspot=0.)

tlags = np.arange(0, 20, 0.1)

In [7]:
TSAMP = 0.1     # time sampling for lightcurve simulations
NSIM = 300      # number of lightcurve simulations to average over

t0_num = time.time()

nk = NumericalKernel(theta_numerical, tsim=TSIM, tsamp=TSAMP, nsim=NSIM, verbose=False)

acf_num = nk.kernel(tlags)
tf_num = time.time() - t0_num

print(f"numerical computation time: {tf_num:.3f}")

In [12]:
# AnalyticKernel receives nspot_rate; resolve_hparam computes
# sigma_k = sqrt(nspot_rate) * (1 - fspot) * alpha_max^2 internally.
t0_ana = time.time()

ak = AnalyticKernel(theta_analytic, n_harmonics=3, n_lat=64,
                    lat_range=(-np.pi/2, np.pi/2), quadrature="trapezoid")

acf_ana = ak.kernel(tlags)
acf_ana = acf_ana / acf_ana[0]  # normalize
tf_ana = time.time() - t0_ana

print(f"analytic computation time: {tf_ana:.3f}")

In [13]:
plt.plot(tlags, acf_num, label="Numerical ACF")
plt.plot(tlags, acf_ana, label="Analytical ACF")
for ii in range(1, 10):
    plt.axvline(ii * theta_analytic["peq"], color='gray', ls='--', alpha=0.6)
plt.axhline(0, color='gray', ls='--', alpha=0.6)
plt.xlabel("Time lag", fontsize=20)
plt.ylabel("Autocorrelation function", fontsize=20)
plt.legend(loc="upper right", fontsize=18)
plt.xlim(min(tlags), max(tlags))
plt.show()